# Centrality bias of O+O jet events, measured like the experiments (R4b, full test)

Step 4 of the plan forward on the tuning page; produced by `oo_bias.py` (branch `oo_rhic_run`). Model
results, not vetted physics results.

## Purpose
STAR and sPHENIX sort O+O events into centrality classes by the charged-particle multiplicity in a forward
(or mid-rapidity) acceptance and normalise each class with the minimum-bias $\langle N_\mathrm{coll}\rangle$ of
that class. In X-SCAPE the hard scattering's momentum is subtracted from the two colliding nucleons
(energy conservation), and 3dMCGlauber's current setting lets the hard pair make no string of its own
(Chun Shen's `09e0a7e`). Both lower the soft activity of jet events, so jet events can land in more peripheral
classes than their geometry deserves. Classified like the experiments, the jet yield then deviates from
$N_\mathrm{coll}$ scaling per class without any medium effect — a selection (centrality) bias that would enter
$R_{AA}$ and $R_{CP}$. The vacuum closure (R4, `closure_oo.ipynb`) showed that at fixed geometry the yield
per hard scattering is unchanged; the proxy studies (R4b part 1) estimated the bias from 3dMCGlauber's strings
without hydro. This notebook measures it with full events: MUSIC hydro and iSS particlization, the
multiplicity counted as the detectors count it.

## Method
- **Events** (O+O 200 GeV, `config/OO_MCGlauber_MUSIC_isr_rhictune.xml`, $b$ 0–10 fm as 3dMCGlauber samples
  it, MUSIC + iSS in the job with one sample per event — the physical fluctuations, the jet **in vacuum**
  (`--vacuum-jet`: Matter `in_vac`, no LBT), so the jet yield itself is not modified):
  - `mb`, the minimum-bias reference: the ISR chain without the subtraction and with the veto off, counting
    only the bulk hadrons. With neither, the chain makes the same strings as minimum-bias events at the same
    $N_\mathrm{coll}$ and samples the same geometry (closure in `closure_oo.ipynb`, part 1d: $N_\mathrm{coll}$,
    $N_\mathrm{part}$ and $b$ distributions identical, string proxies within 1–2 %), so these are minimum-bias
    soft events with exactly the MUSIC and iSS settings of the jet events.
  - `jets`: the production settings (subtraction, veto on); the estimators count bulk + the hard event's hadrons.
  - `jets_noveto`: the same with 3dMCGlauber's hard-pair string veto off.
  Jet events in $\hat p_T$ windows 9–11, 18–23, 30–40, 40–55 GeV, each weighted with its $N_\mathrm{coll}$ (the
  generator puts exactly one hard scattering in every event; its probability is proportional to $N_\mathrm{coll}$).
- **Estimators** (charged particles): STAR's EPD 2.1 < |η| < 5.1 and mid-rapidity |η| < 0.5 / < 1
  ($p_T$ > 0.2 GeV); sPHENIX's MBD 3.51 < |η| < 4.61 (sum of both arms; ≥ 1 particle in each arm is the
  minimum-bias coincidence; sPHENIX: 91.4 %).
- **Per class** (percentiles of the minimum-bias multiplicity): the experiment-style
  $R_{AA}^\mathrm{vac} = (f_{\mathrm{jet},c}/f_{\mathrm{MB},c})\,\langle N_\mathrm{coll}\rangle_\mathrm{MB}/
  \langle N_\mathrm{coll}\rangle_{\mathrm{MB},c}$ — 1 means no bias —, the bias factor
  $\langle N_\mathrm{coll}\rangle_{\mathrm{jet},c}/\langle N_\mathrm{coll}\rangle_{\mathrm{MB},c}$, and $R_{CP}$.

In [ ]:
import json, sys
import numpy as np
import matplotlib.pyplot as plt
import tune as T, oo_bias as B
import hstat as H   # statistical errors: one hydro + iSS sample per event (no oversampling), errors from the events
res = json.loads((T.RES / "oo_bias.json").read_text())
mb = B.load("mb")
J = {n: B.load(n) for n in ("jets", "jets_noveto")}
J = {n: v for n, v in J.items() if v}
print(f"minimum bias: {res['n_mb']} events, <N_coll> {res['mb']['ncoll_mean']:.2f}, events with a medium "
      f"{res['mb']['has_medium']:.3f}, MBD coincidence {res['mb']['eps_mbd_coincidence']:.3f}"
      f"±{res['mb']['eps_mbd_coincidence_err']:.3f} (sPHENIX 0.914)")
print("mean charged multiplicity (minimum bias):", {k: round(v, 1) for k, v in res["mb"]["mean_counts"].items()})
for n, r in res["jets"].items():
    print(f"{n}: {r['n']} events; N_coll-weighted: with a medium {r['has_medium_weighted']:.3f}, "
          f"MBD coincidence {r['eps_mbd_coincidence_weighted']:.4f}±{r['eps_mbd_coincidence_weighted_err']:.4f}")
COL = {"jets": "C3", "jets_noveto": "C0"}
LAB = {"jets": "jets, production (subtraction, veto on)", "jets_noveto": "jets, veto off"}

## The estimator multiplicities
Left: the minimum-bias distributions (class boundaries from them); right: the mean multiplicity against
$N_\mathrm{coll}$ for minimum-bias and jet events. At fixed $N_\mathrm{coll}$ a lower jet-event multiplicity in an
estimator means the jet events slide to more peripheral classes there.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(17, 8))
for col, (e, lab) in enumerate((("epd", "EPD 2.1<|η|<5.1"), ("mbd", "MBD 3.51<|η|<4.61"), ("mid10", "|η|<1, pT>0.2"))):
    ax = axs[0, col]
    b = np.arange(-0.5, max(mb[f"n_{e}"].max(), 1) + 2, max(1, int(mb[f"n_{e}"].max() / 80)))
    H.step(ax, mb[f"n_{e}"], b, color="k", label="minimum bias")
    for n, d in J.items():
        H.step(ax, d[f"n_{e}"], b, w=d["glauber_ncoll"].astype(float), color=COL[n], label=LAB[n] + " (N_coll-weighted)")
    key = "mbd" if e == "mbd" else e
    for c in res["jets"][next(iter(res["jets"]))]["all"][key]["cuts"][:-1]:
        ax.axvline(c, color="0.75", lw=0.6)
    ax.set_yscale("log"); ax.set_xlabel(f"charged particles in {lab}"); ax.legend(fontsize=7)
    ax = axs[1, col]
    edges = np.array([1, 2, 3, 5, 9, 16, 26, 41]); xc = 0.5 * (edges[:-1] + edges[1:] - 1)
    for d, c, l, kn in [(mb, "k", "minimum bias", "glauber_ncoll")] + [(J[n], COL[n], LAB[n], "glauber_ncoll") for n in J]:
        H.points(ax, xc, *H.profile(d[kn], d[f"n_{e}"], edges), color=c, label=l)
    ax.set_xscale("log"); ax.set_xlabel(r"$N_\mathrm{coll}$"); ax.set_ylabel(f"mean multiplicity in {lab}"); ax.legend(fontsize=7)
fig.tight_layout()

## The selection bias per class
Experiment-style $R_{AA}^\mathrm{vac}$ per class (the jet is in vacuum: 1 = no bias) for STAR's EPD and
mid-rapidity classes and sPHENIX's MBD classes, all $\hat p_T$ windows together. Error bars statistical, from both
samples (the $N_\mathrm{coll}$-weighted jet events and the minimum-bias events, `hstat.raa_vac`): every event has one
hydro and one iSS sample, so the events are independent.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, (est, classes, title) in zip(axs, (("epd", res["star_classes"], "STAR EPD classes"),
                                           ("mid10", res["star_classes"], "mid-rapidity |η|<1 classes"),
                                           ("mbd", res["sph_classes"], "sPHENIX MBD classes"))):
    labs = [f"{lo}-{hi}%" for lo, hi in classes]
    for n, r in res["jets"].items():
        a = r["all"][est]
        ax.errorbar(range(len(labs)), a["raa_vac"], a["raa_err"], fmt="o-", color=COL[n], label=LAB[n], capsize=2)
    ax.axhline(1, color="k", lw=0.6); ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs)
    ax.set_ylabel(r"experiment-style $R_{AA}^\mathrm{vac}$"); ax.set_title(title, fontsize=9); ax.legend(fontsize=7)
fig.tight_layout()
for n, r in res["jets"].items():
    print(n)
    for est, classes in (("epd", res["star_classes"]), ("mid10", res["star_classes"]), ("mbd", res["sph_classes"])):
        a = r["all"][est]
        print(f"   {est:5s} R_AA^vac " + "  ".join(f"{lo}-{hi}% {v:.3f}±{e:.3f}" for (lo, hi), v, e in zip(classes, a["raa_vac"], a["raa_err"])))
        print(f"         bias factor " + "  ".join(f"{v:.3f}±{e:.3f}" for v, e in zip(a["bias"], a["bias_err"]))
              + "   <N_coll>_MB,c " + " ".join(f"{v:.2f}" for v in a["ncoll_mb"]))
    m = r["all"]["mbd"]["rcp"]; e = r["all"]["epd_sph"]["rcp"]
    print("   R_CP " + ";  ".join(f"{k}: MBD {m[k][0]:.3f}±{m[k][1]:.3f}, EPD {e[k][0]:.3f}±{e[k][1]:.3f}" for k in m))

## Dependence on the hard scale
The same per $\hat p_T$ window: the subtraction takes more momentum at higher $\hat p_T$, but the proxy found the
forward bias nearly flat (the extra momentum comes from beyond the estimators' acceptance).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, (est, classes) in zip(axs, (("mbd", res["sph_classes"]), ("epd", res["star_classes"]))):
    labs = [f"{lo}-{hi}%" for lo, hi in classes]
    r = res["jets"].get("jets")
    if r is None: continue
    for (wk, rw), c in zip(r["per_window"].items(), ("C0", "C1", "C2", "C3")):
        a = rw[est]
        ax.errorbar(np.arange(len(labs)) + 0.05 * (list(r["per_window"]).index(wk) - 1.5), a["raa_vac"], a["raa_err"],
                    fmt="o-", ms=3, color=c, label=rf"$\hat p_T$ {wk} GeV", capsize=2)
    ax.axhline(1, color="k", lw=0.6); ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs)
    ax.set_ylabel(r"$R_{AA}^\mathrm{vac}$ (production settings)"); ax.set_title(f"{est.upper()} classes", fontsize=9); ax.legend(fontsize=7)
fig.tight_layout()

## Against the jet $p_T$ (reconstructed jets, `oo_bias_jets.py`)
The same events and classes, the yield per class from reconstructed jets: anti-$k_T$ $R=0.4$, $|\eta_\mathrm{jet}|<0.7$
(sPHENIX's), on the hard event's hadrons (the jet is in vacuum, so there is no background to subtract). The files
hold charged hadrons only, so these are **charged-particle jets**: their $p_T$ is roughly 0.6–0.7 of a calorimeter
jet's, sPHENIX's 19–60 GeV ≈ 12–40 GeV here. Each jet counts with its event's $N_\mathrm{coll}\times\sigma_k/N_k$. Per
$\hat p_T$ window (coloured) the jet $p_T$ range is the window's own; the σ-weighted sum over the windows (grey) is not
inclusive (the windows leave gaps at 11–18 and 23–30 GeV), and where a lower window's tail feeds a bin its few
high-weight events dominate the error. Still selection only: no energy loss.

In [ ]:
jp = json.loads((T.RES / "oo_bias_jets.json").read_text())
pe = np.array(jp["pt_edges"]); pc = 0.5 * (pe[1:] + pe[:-1])
fig, axs = plt.subplots(1, 2, figsize=(14, 4.4), sharey=True)
for ax, key, title in ((axs[0], "mbd", "sPHENIX MBD classes"), (axs[1], "epd_sph", "EPD, the same classes")):
    E_ = jp["estimators"][key]
    a = np.array(E_["all"]["rcp"]["0-20/60-80"])
    H.points(ax, pc, a[:, 0], a[:, 1], fmt="s", color="0.6", label="all windows (σ-weighted, not inclusive)")
    for k, (w, r) in enumerate(E_["per_window"].items()):
        v = np.array(r["rcp"]["0-20/60-80"]); n = np.array(r["n_jets"])
        m = n > 300
        H.points(ax, pc[m] * (1 + 0.012 * (k - 1.5)), v[m, 0], v[m, 1], fmt="o-", color=f"C{k}", label=rf"$\hat p_T$ {w} GeV")
    ax.axhspan(0.62, 0.68, color="k", alpha=0.1, label="sPHENIX measured (full jets 19–60 GeV)")
    ax.axhline(1, color="k", lw=0.6); ax.set_xscale("log"); ax.set_ylim(0.4, 1.3)
    ax.set_xlabel(r"charged-jet $p_T$ [GeV] (R = 0.4, |η| < 0.7)"); ax.set_title(title, fontsize=9)
axs[0].set_ylabel(r"selection-only $R_{CP}$(0-20/60-80)"); axs[0].legend(fontsize=7)
fig.tight_layout()
for key in ("mbd", "epd_sph"):
    for w, r in jp["estimators"][key]["per_window"].items():
        print(f"{key:7s} pTHat {w:5s}: " + "  ".join(f"{pe[i]:g}-{pe[i + 1]:g} {v:.2f}±{e:.2f}"
                                               for i, ((v, e), n) in enumerate(zip(r["rcp"]["0-20/60-80"], r["n_jets"])) if n > 300))

## Comparison with the proxy and with sPHENIX
Selection-only $R_{CP}$(0–20/60–80) in sPHENIX's MBD classes: the full events against the string proxy without
and with negative-binomial fluctuations (`centrality_proxy.py`, `closure_oo.ipynb`), and against sPHENIX's
Angantyr estimate (< 10 %) and its measured $R_{CP}$ (≈ 0.62–0.68, read from sPH-CONF-JET-2026-02 Fig. 7).

In [ ]:
rows = []   # (label, value, statistical error)
for n, r in res["jets"].items():
    v, e = r["all"]["mbd"]["rcp"]["0-20/60-80"]; rows.append((f"full events: {LAB[n]}", v, e))
try:
    sm = json.loads((T.RES / "centrality_proxy_smear.json").read_text())
    rows.append(("proxy, no fluctuations (production settings; closure_oo.ipynb)", 0.74, 0.01))
    band = [r["mbd"]["jet_sub"] for r in sm["grid"] if 0.85 <= r["eps_trig_mb"] <= 0.94]
    v = [b["rcp_0_20_60_80"] for b in band]; e = [b["rcp_0_20_60_80_err"] for b in band]
    rows.append((f"proxy, NBD-smeared in sPHENIX's efficiency band ({min(v):.2f}-{max(v):.2f})", np.mean(v), max(e)))
except FileNotFoundError:
    pass
rows.append(("sPHENIX Angantyr estimate (bias < 10 %)", 0.92, 0))
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.barh(range(len(rows)), [v for _, v, _ in rows], xerr=[e for _, _, e in rows], capsize=3,
        color=["C3", "C0", "0.6", "0.4", "C2"][:len(rows)])
ax.axvspan(0.62, 0.68, color="k", alpha=0.12, label="sPHENIX measured R_CP (0-20/60-80)")
ax.axvline(1, color="k", lw=0.6); ax.set_yticks(range(len(rows))); ax.set_yticklabels([l for l, _, _ in rows], fontsize=8)
ax.set_xlim(0.5, 1.15); ax.set_xlabel(r"selection-only $R_{CP}$(0-20/60-80), MBD classes"); ax.legend(fontsize=7)
fig.tight_layout()
for l, v, e in rows: print(f"{v:.3f}±{e:.3f}  {l}")

## Results and what they mean
2026-11-23: 49,996 minimum-bias and 50,000 jet events; the jet in vacuum (`--vacuum-jet`), so **everything here is
selection bias, no energy loss**; statistical errors from both samples. Model results, not vetted physics results.

1. **sPHENIX's MBD classes.** $R_{AA}^\mathrm{vac}$ = 0.958 ± 0.009 / 1.022 ± 0.014 / 1.027 ± 0.017 / 1.093 ± 0.019
   / 1.180 ± 0.021 (0–20 … 80–100 %): $R_{CP}$(0–20/60–80) = **0.876 ± 0.019**, (0–20/40–60) = 0.932 ± 0.020. A bias of
   ≈ 12 %: more than sPHENIX's Angantyr estimate (< 10 %), well short of the measured 0.62–0.68. Factorised, the data
   would be this bias times a suppression of ≈ 0.74; the bias is ≈ 30 % of the suppression (in log).
2. **It grows with the hard scale**: $R_{CP}$ 0.95 ± 0.03 / 0.87 ± 0.03 / 0.84 ± 0.03 / 0.81 ± 0.03 for $\hat p_T$
   9–11 / 18–23 / 30–40 / 40–55 GeV (energy conservation: the subtraction takes more from the strings). For
   reconstructed charged jets in sPHENIX's range (12–40 GeV, windows 30–40 and 40–55) it is 0.80–0.86, flat in jet
   $p_T$ within a window; the highest jet-$p_T$ bin of each window is lower (0.63–0.69 ± 0.09–0.10, a 1.5–2σ hint).
3. **STAR's EPD is less biased**: $R_{AA}^\mathrm{vac}$ ≈ 1 in 0–60 %, 1.06 in 60–80 %, 0.78 in 80–100 %;
   $R_{CP}$(0–20/60–80) 0.951 ± 0.020 in sPHENIX's classes. Mid-rapidity is no estimator for jet events (the jet's own
   particles: $R_{AA}^\mathrm{vac}$ 1.59 in 0–10 % to 0.015 in 80–100 %).
4. **Full events against the proxy**: 0.876 against 0.74 (string proxy, no fluctuations) and 0.82 (NBD-smeared):
   fluctuations and the jet events' own forward hadrons dilute the bias.

**Caveats.** The model's minimum-bias MBD coincidence is 0.847 ± 0.002 (sPHENIX 0.914; jet events 0.969) and its bulk
is too low, most of all peripherally (`oo_bulk_comparison.ipynb`), exactly where the classes decide $R_{CP}$: the
systematic error is larger than the statistical one. The bias depends on how 3dMCGlauber and the subtraction share
the energy between the hard scattering and the remnants, one generator choice. **Next**: the same 50k events with the
jet in the medium (`jets_quench`, started 2026-11-23), so that $R_{CP}$(quenched)/$R_{CP}$(vacuum) separates
quenching from selection.